# 12 WLASL2000 Live Screen / YouTube / Zoom Inference

## What this notebook does

This notebook tests the deployed WLASL2000 model on live screen video, such as:

```text
YouTube sign video
Zoom / Teams / Google Meet signer tile
Browser video
Any video playing on your screen
```

## Pipeline

```text
screen capture region
→ MediaPipe keypoints
→ rolling 60-frame buffer
→ WLASL2000 prediction
→ hand activity check
→ confidence + stability logic
→ accepted / uncertain / waiting output
```

Use this only for videos or meetings where you have permission to process the visual content.

# 1. Install screen capture dependency if needed

In [57]:
import sys
import subprocess

try:
    import mss
    print("mss already installed.")
except ImportError:
    print("Installing mss...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mss"])
    import mss
    print("mss installed.")

mss already installed.


# 2. Import libraries

In [58]:
from pathlib import Path
from collections import Counter, deque
import json
import time
import warnings

import cv2
import mediapipe as mp
import mss
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning)

# 3. Load deployment config and label map

In [59]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DEPLOY_DIR = PROJECT_ROOT / "app" / "models" / "ASL" / "WLASL2000"
CONFIG_FILE = DEPLOY_DIR / "wlasl2000_deployment_config.json"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_PATH = Path(config["model_path"])
LABEL_MAP_PATH = Path(config["label_map_path"])

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    raw_label_map = json.load(f)

id_to_gloss = {int(k): v["gloss"] for k, v in raw_label_map.items()}
rules = config["confidence_rules"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config:", CONFIG_FILE)
print("Model:", MODEL_PATH)
print("Label map:", LABEL_MAP_PATH)
print("Device:", device)
print("Rules:")
print(json.dumps(rules, indent=4))

Config: E:\Be_My_Ear\app\models\ASL\WLASL2000\wlasl2000_deployment_config.json
Model: E:\Be_My_Ear\app\models\ASL\WLASL2000\selected_wlasl2000_model.pt
Label map: E:\Be_My_Ear\app\models\ASL\WLASL2000\asl_wlasl2000_labels.json
Device: cuda
Rules:
{
    "mode": "auto_understanding_with_uncertainty",
    "auto_accept_confidence": 0.45,
    "uncertain_min_confidence": 0.25,
    "reject_below": 0.25,
    "top1_top2_margin": 0.05,
    "stability_window_count": 3,
    "min_repeated_predictions": 2,
    "output_top_k": 5,
    "auto_speak_only_if_stable": true,
    "show_uncertainty_message": true
}


# 4. Load normalisation stats

In [60]:
def find_norm_stats_file():
    candidates = []

    if "norm_stats_path" in config:
        candidates.append(Path(config["norm_stats_path"]))

    model_dir = PROJECT_ROOT / "models" / "ASL" / "WLASL2000"
    selected_name = config.get("selected_model_name", "").lower()

    if "light v3" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v3_two_stage_finetuned_from_wlasl1000_train_norm_stats.npz")

    if "light v2" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v2_finetuned_from_wlasl1000_train_norm_stats.npz")

    candidates.extend(sorted(model_dir.glob("*norm_stats*.npz")))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Could not find WLASL2000 norm stats .npz file.")

NORM_STATS_FILE = find_norm_stats_file()
stats = np.load(NORM_STATS_FILE)

train_mean = stats["mean"].astype(np.float32)
train_std = stats["std"].astype(np.float32)

print("Norm stats:", NORM_STATS_FILE)
print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

Norm stats: E:\Be_My_Ear\models\ASL\WLASL2000\wlasl2000_light_v2_finetuned_from_wlasl1000_train_norm_stats.npz
Mean shape: (258,)
Std shape: (258,)


# 5. Define and load model

In [61]:
class BiGRUAttentionDeploy(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        bi_hidden = hidden_size * 2

        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)

        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)

        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)

checkpoint = torch.load(MODEL_PATH, map_location=device)

num_classes = int(config["num_classes"])
input_size = int(config["input_shape"][1])
hidden_size = int(checkpoint.get("hidden_size", 320))
num_layers = int(checkpoint.get("num_layers", 2))
dropout = float(checkpoint.get("dropout", 0.35))

model = BiGRUAttentionDeploy(input_size, hidden_size, num_classes, num_layers, dropout).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded model:", config["selected_model_name"])
print("Architecture:", checkpoint.get("architecture", "BiGRUAttentionDeploy"))

Loaded model: WLASL2000 Light V2 Fine-tuned From WLASL1000
Architecture: BiGRUAttentionLightV2


# 6. Setup MediaPipe keypoint extraction

In [62]:
mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = int(config["sequence_length"])
BASE_FEATURE_SIZE = int(config["base_keypoint_shape"][1])
INPUT_SIZE = int(config["input_shape"][1])

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4

def extract_landmarks_from_results(results):
    left_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten() if results.left_hand_landmarks else np.zeros(LEFT_HAND_SIZE, dtype=np.float32)
    right_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten() if results.right_hand_landmarks else np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten() if results.pose_landmarks else np.zeros(POSE_SIZE, dtype=np.float32)
    return np.concatenate([left_hand, right_hand, pose]).astype(np.float32)

def process_frame_to_keypoints(frame_bgr, holistic):
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image_rgb.flags.writeable = False
    results = holistic.process(image_rgb)
    return extract_landmarks_from_results(results)

print("MediaPipe ready.")

MediaPipe ready.


# 7. Prediction, activity, and decision helpers

In [63]:
def prepare_model_input(keypoint_sequence):
    keypoint_sequence = np.asarray(keypoint_sequence, dtype=np.float32)

    if keypoint_sequence.shape != (SEQUENCE_LENGTH, BASE_FEATURE_SIZE):
        raise ValueError(f"Expected {(SEQUENCE_LENGTH, BASE_FEATURE_SIZE)}, got {keypoint_sequence.shape}")

    normalised = (keypoint_sequence - train_mean.reshape(1, -1)) / (train_std.reshape(1, -1) + 1e-6)

    velocity = np.zeros_like(normalised, dtype=np.float32)
    velocity[1:] = normalised[1:] - normalised[:-1]

    features = np.concatenate([normalised, velocity], axis=1).astype(np.float32)
    return torch.tensor(features, dtype=torch.float32).unsqueeze(0)

def predict_keypoint_sequence(keypoint_sequence, top_k=5):
    x = prepare_model_input(keypoint_sequence).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()

    top_ids = np.argsort(probs)[-top_k:][::-1]

    top_predictions = [
        {
            "label_id": int(label_id),
            "gloss": id_to_gloss.get(int(label_id), str(label_id)),
            "probability": float(probs[label_id])
        }
        for label_id in top_ids
    ]

    top1 = top_predictions[0]
    top2_prob = top_predictions[1]["probability"] if len(top_predictions) > 1 else 0.0

    return {
        "top1_label_id": top1["label_id"],
        "top1_gloss": top1["gloss"],
        "top1_confidence": top1["probability"],
        "top1_top2_margin": float(top1["probability"] - top2_prob),
        "top_k": top_predictions
    }

def hand_activity_score(sequence):
    sequence = np.asarray(sequence, dtype=np.float32)
    left_hand = sequence[:, :63]
    right_hand = sequence[:, 63:126]
    hands = np.concatenate([left_hand, right_hand], axis=1)

    non_zero_ratio = np.mean(np.abs(hands) > 1e-6)
    movement = np.mean(np.abs(hands[1:] - hands[:-1]))
    active = non_zero_ratio > 0.05 and movement > 0.002

    return {
        "non_zero_ratio": float(non_zero_ratio),
        "movement": float(movement),
        "active": bool(active)
    }

def decision_from_prediction(prediction, recent_predictions=None):
    confidence = prediction["top1_confidence"]
    margin = prediction["top1_top2_margin"]

    auto_accept_confidence = float(rules.get("auto_accept_confidence", 0.45))
    uncertain_min_confidence = float(rules.get("uncertain_min_confidence", 0.25))
    required_margin = float(rules.get("top1_top2_margin", 0.05))
    min_repeated = int(rules.get("min_repeated_predictions", 1))
    stability_count = int(rules.get("stability_window_count", 3))

    stable = False

    if recent_predictions is not None and len(recent_predictions) >= min_repeated:
        last_items = list(recent_predictions)[-stability_count:]
        glosses = [item["top1_gloss"] for item in last_items]
        stable = Counter(glosses)[prediction["top1_gloss"]] >= min_repeated
    else:
        stable = True

    if confidence >= auto_accept_confidence and margin >= required_margin and stable:
        status = "accepted"
        message = f"Detected sign: {prediction['top1_gloss']}"
    elif confidence >= uncertain_min_confidence:
        status = "uncertain"
        alternatives = ", ".join([item["gloss"] for item in prediction["top_k"]])
        message = f"I think this may mean: {prediction['top1_gloss']} | Alternatives: {alternatives}"
    else:
        status = "repeat"
        message = "I am not sure. Please sign again slowly."

    return {
        "status": status,
        "stable": stable,
        "message": message,
        "top1_gloss": prediction["top1_gloss"],
        "confidence": confidence,
        "margin": margin
    }

# 8. Preview monitors

In [64]:
with mss.mss() as sct:
    for i, monitor in enumerate(sct.monitors):
        print(f"Monitor {i}: {monitor}")

print("\nMonitor 1 is usually your main screen.")

Monitor 0: {'left': -1920, 'top': 0, 'width': 3840, 'height': 1080}
Monitor 1: {'left': 0, 'top': 0, 'width': 1920, 'height': 1080, 'is_primary': True, 'name': 'Generic PnP Monitor', 'unique_id': '\\\\?\\DISPLAY#SAM71DC#5&e62ee82&0&UID4352#{e6f07b5f-ee97-4a90-b076-33f57bf4eaa7}'}
Monitor 2: {'left': -1920, 'top': 0, 'width': 1920, 'height': 1080, 'is_primary': False, 'name': 'Generic PnP Monitor', 'unique_id': '\\\\?\\DISPLAY#CMN1521#4&80abe25&0&UID8388688#{e6f07b5f-ee97-4a90-b076-33f57bf4eaa7}'}

Monitor 1 is usually your main screen.


C:\Users\Coding\AppData\Local\Temp\ipykernel_15176\2267723920.py:1: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


# 9. Set screen capture region

In [65]:
# Change this region to crop the YouTube / Zoom / meeting signer tile.
# Coordinates are in screen pixels.

SCREEN_REGION = {
    "top": 120,
    "left": 120,
    "width": 640,
    "height": 480
}

print("Screen region:")
print(SCREEN_REGION)

Screen region:
{'top': 120, 'left': 120, 'width': 640, 'height': 480}


# 10. Preview selected screen region

In [66]:
def capture_screen_region(region):
    with mss.mss() as sct:
        screenshot = np.array(sct.grab(region))
    return cv2.cvtColor(screenshot, cv2.COLOR_BGRA2BGR)

preview_frame = capture_screen_region(SCREEN_REGION)

print("Preview shape:", preview_frame.shape)
print("If this is not the signer/video area, change SCREEN_REGION and run again.")

cv2.imshow("Screen Region Preview - Press any key to close", preview_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

Preview shape: (480, 640, 3)
If this is not the signer/video area, change SCREEN_REGION and run again.


C:\Users\Coding\AppData\Local\Temp\ipykernel_15176\675988849.py:2: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


# 11. Display helpers

In [67]:
def draw_text_block(frame, lines, x=20, y=35, line_height=28):
    h, w = frame.shape[:2]
    box_height = min(h - 20, 20 + line_height * len(lines))

    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (w - 10, box_height + 10), (0, 0, 0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)

    for i, line in enumerate(lines):
        cv2.putText(
            frame,
            str(line),
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.62,
            (0, 255, 0),
            1,
            cv2.LINE_AA
        )

def format_prediction_lines(prediction, decision, accepted_signs, activity=None):
    top5 = ", ".join([item["gloss"] for item in prediction["top_k"]])
    if len(top5) > 70:
        top5 = top5[:70] + "..."

    lines = [
        f"Status: {decision['status'].upper()}",
        f"Top-1: {prediction['top1_gloss']} ({prediction['top1_confidence']:.2f})",
        f"Stable: {decision['stable']} | Margin: {prediction['top1_top2_margin']:.2f}",
        f"Top-5: {top5}"
    ]

    if activity is not None:
        lines.append(f"Hand activity: visible={activity['non_zero_ratio']:.3f}, move={activity['movement']:.4f}")

    if accepted_signs:
        lines.append("Accepted: " + " ".join(accepted_signs[-6:]))

    lines.append("Q=quit | C=clear")
    return lines

def should_accept_new_sign(gloss, accepted_signs, last_accept_time, min_seconds=1.5):
    now = time.time()

    if accepted_signs and accepted_signs[-1] == gloss:
        return False

    if now - last_accept_time < min_seconds:
        return False

    return True

In [68]:
def update_locked_answer(prediction, decision):
    """
    Keeps a detected sign stable on screen so it does not jump immediately.
    """

    global LOCKED_SIGN, LOCKED_CONFIDENCE, LOCKED_AT

    now = time.time()

    # If no accepted prediction, keep old locked sign until it expires
    if prediction is None or decision is None:
        if LOCKED_SIGN is not None and now - LOCKED_AT <= LOCK_SECONDS:
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True
        return None, 0.0, False

    # Only lock accepted predictions
    if decision["status"] != "accepted":
        if LOCKED_SIGN is not None and now - LOCKED_AT <= LOCK_SECONDS:
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True
        return None, 0.0, False

    new_sign = prediction["top1_gloss"]
    new_confidence = prediction["top1_confidence"]

    # If nothing is locked, lock current accepted sign
    if LOCKED_SIGN is None:
        LOCKED_SIGN = new_sign
        LOCKED_CONFIDENCE = new_confidence
        LOCKED_AT = now
        return LOCKED_SIGN, LOCKED_CONFIDENCE, True

    # If same sign, refresh lock
    if new_sign == LOCKED_SIGN:
        LOCKED_CONFIDENCE = max(LOCKED_CONFIDENCE, new_confidence)
        LOCKED_AT = now
        return LOCKED_SIGN, LOCKED_CONFIDENCE, True

    # If different sign appears too soon, ignore it unless very confident
    if now - LOCKED_AT <= LOCK_SECONDS:
        if new_confidence >= MIN_SWITCH_CONFIDENCE:
            LOCKED_SIGN = new_sign
            LOCKED_CONFIDENCE = new_confidence
            LOCKED_AT = now
        return LOCKED_SIGN, LOCKED_CONFIDENCE, True

    # Lock expired, accept new sign
    LOCKED_SIGN = new_sign
    LOCKED_CONFIDENCE = new_confidence
    LOCKED_AT = now

    return LOCKED_SIGN, LOCKED_CONFIDENCE, True

In [69]:
import cv2
import mss
import numpy as np


# ============================================================
# Select screen capture region with mouse
# ============================================================

def capture_full_monitor(monitor_index=1):
    """
    Captures the full selected monitor.
    monitor_index:
    - 1 = main screen usually
    - 2 = second monitor usually
    """
    with mss.mss() as sct:
        monitor = sct.monitors[monitor_index]
        screenshot = np.array(sct.grab(monitor))

    # mss gives BGRA, convert to BGR for OpenCV
    frame_bgr = cv2.cvtColor(screenshot, cv2.COLOR_BGRA2BGR)

    return frame_bgr, monitor


def select_screen_region(monitor_index=1):
    """
    Lets you select a screen region using your mouse.

    Controls:
    - Drag mouse to select area
    - Press ENTER or SPACE to confirm
    - Press C to cancel and reselect
    """
    full_frame, monitor = capture_full_monitor(monitor_index)

    print("Select the signer/video area.")
    print("Drag with mouse, then press ENTER or SPACE.")
    print("Press C to cancel/reselect.")

    roi = cv2.selectROI(
        "Select Screen Region - ENTER/SPACE to confirm, C to cancel",
        full_frame,
        showCrosshair=True,
        fromCenter=False
    )

    cv2.destroyWindow("Select Screen Region - ENTER/SPACE to confirm, C to cancel")

    x, y, w, h = roi

    if w == 0 or h == 0:
        raise ValueError("No region selected. Run the cell again and drag a box around the signer/video.")

    # Convert ROI coordinate to real screen coordinate
    screen_region = {
        "left": int(monitor["left"] + x),
        "top": int(monitor["top"] + y),
        "width": int(w),
        "height": int(h)
    }

    return screen_region


# Choose monitor
# 1 = main monitor
# 2 = second monitor if you have one
MONITOR_INDEX = 1

SCREEN_REGION = select_screen_region(monitor_index=MONITOR_INDEX)

print("Selected SCREEN_REGION:")
print(SCREEN_REGION)

C:\Users\Coding\AppData\Local\Temp\ipykernel_15176\60913018.py:17: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


Select the signer/video area.
Drag with mouse, then press ENTER or SPACE.
Press C to cancel/reselect.
Selected SCREEN_REGION:
{'left': 61, 'top': 210, 'width': 1265, 'height': 707}


In [70]:
preview_frame = capture_screen_region(SCREEN_REGION)

print("Preview frame shape:", preview_frame.shape)

cv2.imshow("Selected Region Preview - Press any key to close", preview_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

Preview frame shape: (707, 1265, 3)


C:\Users\Coding\AppData\Local\Temp\ipykernel_15176\675988849.py:2: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


# 12. Run live screen inference

LOCK_SECONDS = 3.5
same prediction 3 times within 5 seconds
average confidence check
best confidence check
average margin check
stronger anti-wrong-lock filtering

In [ ]:
WINDOW_NAME = "Be My Ear - Live Screen Inference"

PREDICT_EVERY_N_FRAMES = 8
MIN_SECONDS_BETWEEN_ACCEPTED_SIGNS = 1.5
SHOW_ACTIVITY_DEBUG = True

keypoint_buffer = deque(maxlen=SEQUENCE_LENGTH)
recent_predictions = deque(maxlen=int(rules.get("stability_window_count", 3)))

accepted_signs = []
last_accept_time = 0.0
frame_counter = 0

latest_prediction = None
latest_decision = None
latest_activity = None


# ============================================================
# Answer locking system
# ============================================================

LOCKED_SIGN = None
LOCKED_CONFIDENCE = 0.0
LOCKED_AT = 0.0

LOCK_SECONDS = 3.5
MIN_SWITCH_CONFIDENCE = 0.70

# Repeated-prediction lock rule
LOCK_HISTORY_SECONDS = 5.0
MIN_APPEARANCES_TO_LOCK = 3

# Quality checks to stop wrong weak predictions from locking
MIN_AVG_CONFIDENCE_TO_LOCK = 0.35
MIN_BEST_CONFIDENCE_TO_LOCK = 0.45
MIN_AVG_MARGIN_TO_LOCK = 0.05

prediction_history = deque()


def clean_old_prediction_history():
    """
    Removes old predictions outside the lock history time window.
    """

    now = time.time()

    while prediction_history and now - prediction_history[0]["time"] > LOCK_HISTORY_SECONDS:
        prediction_history.popleft()


def add_prediction_to_history(prediction):
    """
    Adds current Top-1 prediction to recent history.
    """

    if prediction is None:
        return

    prediction_history.append({
        "time": time.time(),
        "gloss": prediction["top1_gloss"],
        "confidence": float(prediction["top1_confidence"]),
        "margin": float(prediction["top1_top2_margin"])
    })

    clean_old_prediction_history()


def get_repeated_prediction_candidate():
    """
    Checks whether a sign appeared enough times within the recent time window.

    A sign only becomes a lock candidate if:
    - it appears enough times
    - average confidence is high enough
    - best confidence is high enough
    - average Top-1 vs Top-2 margin is high enough
    """

    clean_old_prediction_history()

    if len(prediction_history) == 0:
        return None, 0.0, 0, "No history"

    grouped = {}

    for item in prediction_history:
        gloss = item["gloss"]

        if gloss not in grouped:
            grouped[gloss] = {
                "count": 0,
                "confidences": [],
                "margins": []
            }

        grouped[gloss]["count"] += 1
        grouped[gloss]["confidences"].append(item["confidence"])
        grouped[gloss]["margins"].append(item["margin"])

    best_gloss = max(grouped, key=lambda g: grouped[g]["count"])
    info = grouped[best_gloss]

    count = info["count"]
    avg_confidence = float(np.mean(info["confidences"]))
    best_confidence = float(np.max(info["confidences"]))
    avg_margin = float(np.mean(info["margins"]))

    reason = (
        f"count={count}, "
        f"avg_conf={avg_confidence:.2f}, "
        f"best_conf={best_confidence:.2f}, "
        f"avg_margin={avg_margin:.2f}"
    )

    if count < MIN_APPEARANCES_TO_LOCK:
        return None, 0.0, count, f"Not enough repeats | {reason}"

    if avg_confidence < MIN_AVG_CONFIDENCE_TO_LOCK:
        return None, 0.0, count, f"Avg confidence too low | {reason}"

    if best_confidence < MIN_BEST_CONFIDENCE_TO_LOCK:
        return None, 0.0, count, f"Best confidence too low | {reason}"

    if avg_margin < MIN_AVG_MARGIN_TO_LOCK:
        return None, 0.0, count, f"Margin too low | {reason}"

    return best_gloss, best_confidence, count, f"Repeated lock passed | {reason}"


def update_locked_answer(prediction, decision):
    """
    Keeps a detected sign stable on screen so it does not jump immediately.

    Lock happens when:
    1. The same Top-1 prediction appears 3 times within 5 seconds
       and passes confidence/margin checks.
    2. The model accepts the prediction directly.
    """

    global LOCKED_SIGN, LOCKED_CONFIDENCE, LOCKED_AT

    now = time.time()

    # Add prediction to temporal history even if not accepted
    if prediction is not None:
        add_prediction_to_history(prediction)

    repeated_sign, repeated_confidence, repeated_count, repeated_reason = get_repeated_prediction_candidate()

    # --------------------------------------------------------
    # Rule A: repeated prediction lock with quality checks
    # --------------------------------------------------------
    if repeated_sign is not None:
        if LOCKED_SIGN is None:
            LOCKED_SIGN = repeated_sign
            LOCKED_CONFIDENCE = repeated_confidence
            LOCKED_AT = now
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True, repeated_reason

        if repeated_sign == LOCKED_SIGN:
            LOCKED_CONFIDENCE = max(LOCKED_CONFIDENCE, repeated_confidence)
            LOCKED_AT = now
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True, repeated_reason

        # If a different repeated sign appears while lock is active,
        # only switch if confidence is strong.
        if now - LOCKED_AT <= LOCK_SECONDS:
            if repeated_confidence >= MIN_SWITCH_CONFIDENCE:
                LOCKED_SIGN = repeated_sign
                LOCKED_CONFIDENCE = repeated_confidence
                LOCKED_AT = now

            return LOCKED_SIGN, LOCKED_CONFIDENCE, True, repeated_reason

        # Lock expired, accept new repeated sign
        LOCKED_SIGN = repeated_sign
        LOCKED_CONFIDENCE = repeated_confidence
        LOCKED_AT = now

        return LOCKED_SIGN, LOCKED_CONFIDENCE, True, repeated_reason

    # --------------------------------------------------------
    # Rule B: if no repeated candidate, keep previous lock
    # --------------------------------------------------------
    if prediction is None or decision is None:
        if LOCKED_SIGN is not None and now - LOCKED_AT <= LOCK_SECONDS:
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Holding previous answer"

        return None, 0.0, False, "No lock"

    # --------------------------------------------------------
    # Rule C: direct accepted prediction lock
    # --------------------------------------------------------
    if decision["status"] != "accepted":
        if LOCKED_SIGN is not None and now - LOCKED_AT <= LOCK_SECONDS:
            return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Holding previous answer"

        return None, 0.0, False, repeated_reason

    new_sign = prediction["top1_gloss"]
    new_confidence = float(prediction["top1_confidence"])

    if LOCKED_SIGN is None:
        LOCKED_SIGN = new_sign
        LOCKED_CONFIDENCE = new_confidence
        LOCKED_AT = now
        return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Accepted directly"

    if new_sign == LOCKED_SIGN:
        LOCKED_CONFIDENCE = max(LOCKED_CONFIDENCE, new_confidence)
        LOCKED_AT = now
        return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Same sign refreshed"

    if now - LOCKED_AT <= LOCK_SECONDS:
        if new_confidence >= MIN_SWITCH_CONFIDENCE:
            LOCKED_SIGN = new_sign
            LOCKED_CONFIDENCE = new_confidence
            LOCKED_AT = now

        return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Locked answer protected"

    LOCKED_SIGN = new_sign
    LOCKED_CONFIDENCE = new_confidence
    LOCKED_AT = now

    return LOCKED_SIGN, LOCKED_CONFIDENCE, True, "Lock expired, new sign accepted"


cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, SCREEN_REGION["width"], SCREEN_REGION["height"])

print("Starting live screen inference.")
print("Press Q in the output window to quit.")
print("Press C in the output window to clear transcript.")
print(f"Lock hold time: {LOCK_SECONDS} seconds")
print(f"Repeated lock: {MIN_APPEARANCES_TO_LOCK} appearances within {LOCK_HISTORY_SECONDS} seconds")
print(f"Quality lock checks: avg_conf >= {MIN_AVG_CONFIDENCE_TO_LOCK}, best_conf >= {MIN_BEST_CONFIDENCE_TO_LOCK}, avg_margin >= {MIN_AVG_MARGIN_TO_LOCK}")

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while True:
        frame = capture_screen_region(SCREEN_REGION)

        keypoints = process_frame_to_keypoints(frame, holistic)
        keypoint_buffer.append(keypoints)

        frame_counter += 1

        # ----------------------------------------------------
        # Predict only when buffer is ready
        # ----------------------------------------------------
        if len(keypoint_buffer) == SEQUENCE_LENGTH and frame_counter % PREDICT_EVERY_N_FRAMES == 0:
            sequence = np.array(keypoint_buffer, dtype=np.float32)
            latest_activity = hand_activity_score(sequence)

            if not latest_activity["active"]:
                latest_prediction = None
                latest_decision = {
                    "status": "waiting",
                    "stable": False,
                    "message": "Waiting for clear hand sign..."
                }

            else:
                latest_prediction = predict_keypoint_sequence(
                    sequence,
                    top_k=int(rules.get("output_top_k", 5))
                )

                recent_predictions.append(latest_prediction)

                latest_decision = decision_from_prediction(
                    latest_prediction,
                    recent_predictions=recent_predictions
                )

                locked_sign, locked_confidence, is_locked, lock_reason = update_locked_answer(
                    latest_prediction,
                    latest_decision
                )

                if is_locked and should_accept_new_sign(
                    locked_sign,
                    accepted_signs,
                    last_accept_time,
                    min_seconds=MIN_SECONDS_BETWEEN_ACCEPTED_SIGNS
                ):
                    accepted_signs.append(locked_sign)
                    last_accept_time = time.time()

        # ----------------------------------------------------
        # Display frame
        # ----------------------------------------------------
        display_frame = frame.copy()

        locked_sign, locked_confidence, is_locked, lock_reason = update_locked_answer(
            latest_prediction,
            latest_decision
        )

        if latest_prediction is not None and latest_decision is not None:
            lines = format_prediction_lines(
                latest_prediction,
                latest_decision,
                accepted_signs,
                activity=latest_activity if SHOW_ACTIVITY_DEBUG else None
            )

            if is_locked:
                lines.insert(
                    1,
                    f"LOCKED ANSWER: {locked_sign} ({locked_confidence:.2f})"
                )
                lines.insert(
                    2,
                    f"Lock reason: {lock_reason}"
                )
            else:
                lines.insert(
                    1,
                    f"No lock: {lock_reason}"
                )

        elif latest_decision is not None and latest_decision.get("status") == "waiting":
            lines = [
                "Status: WAITING",
                latest_decision["message"],
                f"Buffer: {len(keypoint_buffer)}/{SEQUENCE_LENGTH}"
            ]

            if is_locked:
                lines.insert(
                    1,
                    f"LOCKED ANSWER: {locked_sign} ({locked_confidence:.2f})"
                )
                lines.insert(
                    2,
                    f"Lock reason: {lock_reason}"
                )

            if latest_activity is not None and SHOW_ACTIVITY_DEBUG:
                lines.append(
                    f"Hand activity: visible={latest_activity['non_zero_ratio']:.3f}, "
                    f"move={latest_activity['movement']:.4f}"
                )

            if accepted_signs:
                lines.append("Accepted: " + " ".join(accepted_signs[-6:]))

            lines.append("Q=quit | C=clear")

        else:
            lines = [
                f"Collecting frames: {len(keypoint_buffer)}/{SEQUENCE_LENGTH}",
                "Play a video or crop the meeting signer tile.",
                "Q=quit | C=clear"
            ]

        draw_text_block(display_frame, lines)

        cv2.imshow(WINDOW_NAME, display_frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        if key == ord("c"):
            accepted_signs = []
            recent_predictions.clear()
            prediction_history.clear()

            latest_prediction = None
            latest_decision = None
            latest_activity = None

            LOCKED_SIGN = None
            LOCKED_CONFIDENCE = 0.0
            LOCKED_AT = 0.0

            print("Transcript cleared.")

cv2.destroyAllWindows()

print("Live screen inference stopped.")
print("Accepted signs:", accepted_signs)
print("Auto-understanding output:", " ".join(accepted_signs))

Starting live screen inference.
Press Q in the output window to quit.
Press C in the output window to clear transcript.
Lock rule: same sign appears 3 times within 5.0 seconds
Lock hold time: 3.5 seconds


C:\Users\Coding\AppData\Local\Temp\ipykernel_15176\675988849.py:2: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


Live screen inference stopped.
Accepted signs: ['brown', 'bear', 'bird', 'egypt', 'camel', 'wide', 'cat', 'find', 'believe', 'stubborn', 'computer', 'shine', 'alligator', 'deer', 'sunday', 'karate', 'whale', 'boy']
Auto-understanding output: brown bear bird egypt camel wide cat find believe stubborn computer shine alligator deer sunday karate whale boy


# 13. Save live screen session summary

In [72]:
REPORT_DIR = PROJECT_ROOT / "reports" / "inference"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
summary_file = REPORT_DIR / f"live_screen_session_{timestamp}_wlasl2000_summary.json"

summary = {
    "timestamp": timestamp,
    "model": config["selected_model_name"],
    "screen_region": SCREEN_REGION,
    "accepted_signs": accepted_signs,
    "auto_understanding_output": " ".join(accepted_signs),
    "rules": rules,
    "predict_every_n_frames": PREDICT_EVERY_N_FRAMES
}

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Saved summary:", summary_file)

Saved summary: E:\Be_My_Ear\reports\inference\live_screen_session_20260607_173307_wlasl2000_summary.json


# Notes

## For YouTube
Crop the signer area only. Make sure the hands and upper body are visible.

## For Zoom / meetings
Crop only the signer’s video tile, not the full meeting window.

## Limitation
This WLASL2000 model is still isolated-sign recognition. Continuous conversations and full sentence translation will require a later continuous-sign model and NLP sentence builder.